# Qwen3-0.6B Full-Parameter Fine-Tuning

This notebook targets the **Ascend 910B CANN MindSpore image** and follows the upstream Qwen3 MindSpore fine-tuning flow from `MindSpeed-LLM/docs/zh/mindspore/quick_start.md`.

To match the image-bundled source tree and this verification scenario, the notebook inlines commands equivalent to the official scripts:
- `examples/mindspore/qwen3/ckpt_convert_qwen3_hf2mcore.sh`
- `examples/mindspore/qwen3/data_convert_qwen3_instruction.sh`
- `examples/mindspore/qwen3/tune_qwen3_0point6b_4K_full_ms.sh`

**Workflow:**
1. Environment checks
2. Prepare the instruction dataset
3. Verify the bundled MindSpeed-Core-MS/MindSpeed-LLM source tree
4. Convert HF weights to MindSpeed/Mcore format
5. Preprocess the fine-tuning data
6. Launch full-parameter SFT fine-tuning
7. Validate the output checkpoint

> The default parameters are conservative and intended for image validation. For longer training closer to the upstream baseline, increase `SEQ_LENGTH`, `TRAIN_ITERS`, and `MBS` as needed.


## 0. Parameters


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=ImportWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

from pathlib import Path

# ===== Path configuration =====
MINDSPEED_CORE_MS_DEFAULT_PATH = Path('/opt/app-root/share/MindSpeed-Core-MS')
MINDSPEED_CORE_MS_PATH = Path(
    os.environ.get('MINDSPEED_CORE_MS_PATH', str(MINDSPEED_CORE_MS_DEFAULT_PATH))
)
MINDSPEED_LLM_DIR = MINDSPEED_CORE_MS_PATH / 'MindSpeed-LLM'
MINDSPEED_DIR = MINDSPEED_CORE_MS_PATH / 'MindSpeed'
MSADAPTER_DIR = MINDSPEED_CORE_MS_PATH / 'MSAdapter'
MEGATRON_DIR = MINDSPEED_CORE_MS_PATH / 'Megatron-LM'
SET_PATH_SCRIPT = MINDSPEED_CORE_MS_PATH / 'tests' / 'scripts' / 'set_path.sh'

OFFICIAL_CONVERT_SCRIPT = MINDSPEED_LLM_DIR / 'examples' / 'mindspore' / 'qwen3' / 'ckpt_convert_qwen3_hf2mcore.sh'
OFFICIAL_PREPROCESS_SCRIPT = MINDSPEED_LLM_DIR / 'examples' / 'mindspore' / 'qwen3' / 'data_convert_qwen3_instruction.sh'
OFFICIAL_TUNE_SCRIPT = MINDSPEED_LLM_DIR / 'examples' / 'mindspore' / 'qwen3' / 'tune_qwen3_0point6b_4K_full_ms.sh'
CONVERT_CKPT_ENTRY = MINDSPEED_LLM_DIR / 'mindspeed_llm' / 'mindspore' / 'convert_ckpt.py'
assert CONVERT_CKPT_ENTRY.exists(), f'Official MindSpore conversion entry not found: {CONVERT_CKPT_ENTRY}'

HF_MODEL_DIR = Path('/opt/app-root/src/models/Qwen3-0.6B')
WORK_DIR = Path('/opt/app-root/src/Qwen3-0.6B-work-dir')
DATA_DIR = WORK_DIR / 'finetune_dataset'
RAW_DATA_FILE = DATA_DIR / 'alpaca_sample.jsonl'
PROCESSED_DATA_PREFIX = DATA_DIR / 'alpaca'
OUTPUT_DIR = WORK_DIR / 'output' / 'qwen3_0.6b_finetuned'
LOGS_DIR = WORK_DIR / 'logs'
PRECHECK_LOG_DIR = LOGS_DIR / 'preflight'

# ===== Optional: real dataset path =====
ALPACA_PARQUET = Path('/opt/app-root/src/datasets/alpaca/train-00000-of-00001-a09b74b3ef9c3b56.parquet')

# ===== Ascend environment scripts =====
CANN_ENV = '/usr/local/Ascend/cann/set_env.sh'
ATB_ENV = '/usr/local/Ascend/nnal/atb/set_env.sh'

# ===== Repository and model configuration =====
MODEL_SPEC = 'mindspeed_llm.tasks.models.spec.qwen3_spec layer_spec'
MASTER_ADDR = 'localhost'
MASTER_PORT = 6015
NNODES = 1
NODE_RANK = 0
DISTRIBUTED_BACKEND = 'nccl'  # Keep nccl to match the upstream tune_qwen3_0point6b_4K_full_ms.sh behavior.

# ===== Parallelism configuration (must match weight conversion) =====
# Qwen3-0.6B is small enough for TP=1 and PP=1; use both cards for data parallelism.
TP = 1
PP = 1
MBS = 2  # micro-batch=2 fits on each 32G card.

# ===== Weight conversion output (include TP/PP in the path to avoid stale reuse) =====
MCORE_WEIGHTS_DIR = WORK_DIR / 'model_weights' / f'qwen3_mcore_tp{TP}_pp{PP}'

# ===== Training hyperparameters (for 2x 910B 32G) =====
SEQ_LENGTH = 2048  # The upstream quick start example uses 4096; reduce to 2048 for validation.
TRAIN_ITERS = 100  # The upstream quick start example uses 2000; reduce to 100 for validation.
LR = 1.25e-6
MIN_LR = 1.25e-7

# ===== Data preprocessing =====
ENABLE_THINKING = 'true'
HANDLER_NAME = 'AlpacaStyleInstructionHandler'
TOKENIZER_TYPE = 'PretrainedFromHF'
PROMPT_TYPE = 'qwen3'
DATA_PATH = str(PROCESSED_DATA_PREFIX)

print('Configuration loaded')
print(f'  MindSpeed-Core-MS: {MINDSPEED_CORE_MS_PATH}')
print(f'  MindSpeed-LLM:     {MINDSPEED_LLM_DIR}')
print(f'  Official convert script:    {OFFICIAL_CONVERT_SCRIPT}')
print(f'  Official preprocess script: {OFFICIAL_PREPROCESS_SCRIPT}')
print(f'  Official fine-tune script:  {OFFICIAL_TUNE_SCRIPT}')
print(f'  Official conversion entry:  {CONVERT_CKPT_ENTRY}')
print(f'  Model:                      {HF_MODEL_DIR}')
print(f'  Work dir:                   {WORK_DIR}')
print(f'  Dataset: {ALPACA_PARQUET}' if ALPACA_PARQUET.exists() else '  Dataset: built-in sample dataset')
print(f'  TP={TP}, PP={PP}, MBS={MBS}, SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}')
print(f'  Distributed backend: {DISTRIBUTED_BACKEND}')


## Helpers


In [ ]:
import json
import os
import shlex
import subprocess

_SUPPRESS_WARNINGS = 'ignore::DeprecationWarning,ignore::ImportWarning,ignore::UserWarning,ignore::FutureWarning'

def q(value):
    return shlex.quote(str(value))

def run_cmd(cmd, cwd=None, check=True, step_name=None, log_file=None):
    'Run a bash command inside the Ascend environment, keep pipefail semantics, and optionally tee output to a log file.'
    env_parts = ['set -o pipefail', f'source {q(CANN_ENV)}', f'source {q(ATB_ENV)}']
    if SET_PATH_SCRIPT.exists():
        env_parts.append(f'source {q(SET_PATH_SCRIPT)}')
    env_prefix = ' && '.join(env_parts)
    effective_cwd = Path(cwd or WORK_DIR)
    resolved_log_file = Path(log_file) if log_file else None
    wrapped_cmd = cmd
    if resolved_log_file is not None:
        resolved_log_file.parent.mkdir(parents=True, exist_ok=True)
        wrapped_cmd = f'{{
{cmd}
}} 2>&1 | tee {q(resolved_log_file)}'
    full_cmd = f'{env_prefix} && {wrapped_cmd}'
    print(f'$ {cmd}
')
    if resolved_log_file is not None:
        print(f'Log file: {resolved_log_file}
')
    run_env = os.environ.copy()
    run_env['PYTHONWARNINGS'] = _SUPPRESS_WARNINGS
    result = subprocess.run(
        ['bash', '-lc', full_cmd],
        cwd=str(effective_cwd),
        text=True,
        env=run_env,
    )
    if check and result.returncode != 0:
        step_label = f'Step[{step_name}]' if step_name else 'Command'
        log_hint = f', log file: {resolved_log_file}' if resolved_log_file is not None else ''
        raise RuntimeError(f'{step_label} failed with exit code {result.returncode}{log_hint}')
    return result

print('Helper ready: run_cmd()')
print(f'Preflight log dir: {PRECHECK_LOG_DIR}')


## 1. Environment Checks


In [ ]:
import os
import warnings

print('=' * 60)
print('Environment checks')
print('=' * 60)

assert Path(CANN_ENV).exists(), f'Ascend CANN environment script not found: {CANN_ENV}'
assert Path(ATB_ENV).exists(), f'Ascend ATB environment script not found: {ATB_ENV}'
print(f'Expected MindSpeed-Core-MS path: {MINDSPEED_CORE_MS_PATH}')
assert MINDSPEED_CORE_MS_PATH.exists(), f'MindSpeed-Core-MS source tree not found: {MINDSPEED_CORE_MS_PATH}'
assert SET_PATH_SCRIPT.exists(), f'set_path.sh not found: {SET_PATH_SCRIPT}'
for repo_dir in (MINDSPEED_LLM_DIR, MINDSPEED_DIR, MSADAPTER_DIR, MEGATRON_DIR):
    assert repo_dir.exists(), f'Bundled source tree missing: {repo_dir}'
for script_path in (OFFICIAL_CONVERT_SCRIPT, OFFICIAL_PREPROCESS_SCRIPT, OFFICIAL_TUNE_SCRIPT):
    assert script_path.exists(), f'Official example script missing: {script_path}'

with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    warnings.simplefilter('ignore', ImportWarning)
    warnings.simplefilter('ignore', UserWarning)
    warnings.simplefilter('ignore', FutureWarning)
    import mindspore as ms
    import msadapter
    import mindspeed
    import mindspeed_llm

print(f'MindSpore:          {ms.__version__}')
print(f'CANN env script:    {CANN_ENV}')
print(f'ATB env script:     {ATB_ENV}')
print(f'MindSpeed-Core-MS:  {MINDSPEED_CORE_MS_PATH}')
print(f'set_path.sh:        {SET_PATH_SCRIPT}')

device_target = 'unknown'
if hasattr(ms, 'get_context'):
    try:
        device_target = ms.get_context('device_target')
    except Exception:
        pass

nproc = None
hal = getattr(ms, 'hal', None)
if hal is not None and hasattr(hal, 'device_count'):
    try:
        nproc = int(hal.device_count())
    except Exception:
        nproc = None

if not nproc:
    visible_devices = os.environ.get('ASCEND_RT_VISIBLE_DEVICES') or os.environ.get('ASCEND_VISIBLE_DEVICES')
    if visible_devices:
        nproc = len([d for d in visible_devices.split(',') if d.strip()])
    else:
        nproc = int(os.environ.get('RANK_SIZE', '1'))

print(f'Device target:      {device_target}')
print(f'NPU count:          {nproc}')

print(f'
Model directory: {HF_MODEL_DIR}')
assert HF_MODEL_DIR.exists(), f'Model directory not found: {HF_MODEL_DIR}'
required_model_files = [
    HF_MODEL_DIR / 'config.json',
    HF_MODEL_DIR / 'tokenizer.json',
    HF_MODEL_DIR / 'tokenizer_config.json',
]
for required_file in required_model_files:
    assert required_file.exists(), f'Required model file not found: {required_file}'
safetensor_files = sorted(HF_MODEL_DIR.glob('*.safetensors'))
assert safetensor_files, f'No safetensors weight files found: {HF_MODEL_DIR}'
print('Required model files:')
for required_file in required_model_files:
    print(f'  {required_file.name}: OK')
print(f'  safetensors: {len(safetensor_files)} file(s)')

model_files = sorted(HF_MODEL_DIR.glob('*'))
for f in model_files[:5]:
    if f.is_file():
        print(f'  {f.name} ({f.stat().st_size / 1e9:.2f} GB)')
if len(model_files) > 5:
    print(f'  ... total {len(model_files)} files')

py_path_entries = [Path(p) for p in os.environ.get('PYTHONPATH', '').split(':') if p]
expected_entries = [
    MINDSPEED_CORE_MS_PATH / 'msadapter',
    MSADAPTER_DIR,
    MINDSPEED_CORE_MS_PATH / 'msadapter' / 'msa_thirdparty',
    MSADAPTER_DIR / 'msa_thirdparty',
    MINDSPEED_LLM_DIR,
    MEGATRON_DIR,
    MINDSPEED_DIR,
]
print('
PYTHONPATH key entries:')
for entry in expected_entries:
    present = entry in py_path_entries
    print(f'  {entry}: {"OK" if present else "missing"}')
missing_entries = [str(entry) for entry in expected_entries if entry not in py_path_entries]
assert not missing_entries, f'PYTHONPATH is missing required entries: {missing_entries}'

assert nproc >= TP * PP, f'NPU count ({nproc}) < TP*PP ({TP * PP}); reduce PP and retry.'
DP = max(1, nproc // (TP * PP))
GBS = DP * MBS
print(f'
Parallel configuration: TP={TP}, PP={PP}, DP={DP}, GBS={GBS}')

subprocess_env_check = '
'.join([
    "python - <<'PY'",
    'import importlib',
    'import os',
    'import sys',
    'from pathlib import Path',
    '',
    "core_root = Path(os.environ['MINDSPEED_CORE_MS_PATH'])",
    "print(f'sys.executable: {sys.executable}')",
    "print(f'MINDSPEED_CORE_MS_PATH: {core_root}')",
    "for name in ('mindspore', 'msadapter', 'mindspeed', 'mindspeed_llm', 'transformers'):",
    '    module = importlib.import_module(name)',
    "    print(f'{name}: {getattr(module, "__file__", "<namespace>")}')",
    '',
    "cfg = core_root / 'MindSpeed-LLM' / 'configs' / 'checkpoint' / 'model_cfg.json'",
    "print(f'checkpoint_cfg: {cfg} exists={cfg.exists()}')",
    'PY',
    '',
])
run_cmd(
    subprocess_env_check,
    step_name='subprocess-import-check',
    log_file=PRECHECK_LOG_DIR / 'subprocess_import_check.log',
)
print(f'Subprocess environment check log: {PRECHECK_LOG_DIR / "subprocess_import_check.log"}')
print('
Environment checks passed!')


## 2. Prepare the Dataset

Create sample Alpaca-formatted instruction data for fine-tuning flow validation.

To use a real dataset, place a parquet file at `ALPACA_PARQUET` or write a JSONL file to `RAW_DATA_FILE`, one JSON object per line:

```json
{"instruction": "...", "input": "...", "output": "..."}
```


In [ ]:
import warnings

DATA_DIR.mkdir(parents=True, exist_ok=True)

if ALPACA_PARQUET.exists():
    print(f'Loading Alpaca dataset: {ALPACA_PARQUET.name}')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)
        import pandas as pd
        df = pd.read_parquet(ALPACA_PARQUET)

    required_columns = {'instruction', 'input', 'output'}
    missing_columns = required_columns.difference(df.columns)
    assert not missing_columns, f'Parquet dataset is missing required columns: {sorted(missing_columns)}'

    with open(RAW_DATA_FILE, 'w', encoding='utf-8') as f:
        for item in df[['instruction', 'input', 'output']].to_dict('records'):
            item['input'] = item.get('input') or ''
            f.write(json.dumps(item, ensure_ascii=False) + '
')

    print(f'Converted parquet to JSONL: {RAW_DATA_FILE}')
    preview_records = df[['instruction', 'input', 'output']].head(3).to_dict('records')
else:
    print('Alpaca dataset not found, using built-in sample data
')
    sample_data = [
        {'instruction': 'Translate the following sentence into French', 'input': 'The weather is nice today.', 'output': "Il fait beau aujourd'hui."},
        {'instruction': 'Translate the following sentence into Spanish', 'input': 'I like programming.', 'output': 'Me gusta programar.'},
        {'instruction': 'Summarize the following in one sentence', 'input': 'Machine learning is fascinating and widely used in many fields.', 'output': 'Machine learning is widely used across many fields.'},
        {'instruction': 'Rewrite in a more formal tone', 'input': 'Hello, how are you?', 'output': 'Hello, how have you been today?'},
        {'instruction': 'Introduce MindSpore in one sentence', 'input': '', 'output': 'MindSpore is an all-scenario AI computing framework for device, edge, and cloud.'},
        {'instruction': 'List three common sorting algorithms', 'input': '', 'output': 'Three common sorting algorithms are bubble sort, quicksort, and merge sort.'},
        {'instruction': 'Explain what full-parameter fine-tuning means', 'input': '', 'output': 'Full-parameter fine-tuning updates all model weights instead of training only a lightweight adapter.'},
        {'instruction': 'Write a Python function that adds two numbers', 'input': '', 'output': 'def add(a, b):
    return a + b'},
        {'instruction': 'Rewrite in a more concise way', 'input': 'Artificial intelligence is changing the world.', 'output': 'AI is changing the world.'},
        {'instruction': 'What is Ascend 910B?', 'input': '', 'output': 'Ascend 910B is an AI accelerator chip designed by Huawei for deep learning training and inference.'},
    ]
    with open(RAW_DATA_FILE, 'w', encoding='utf-8') as f:
        for item in sample_data:
            f.write(json.dumps(item, ensure_ascii=False) + '
')
    preview_records = sample_data[:3]
    print(f'Sample dataset created: {RAW_DATA_FILE}')
    print(f'{len(sample_data)} samples total')

print('
Data preview:')
for item in preview_records:
    inp = f' {item["input"]}' if item.get('input') else ''
    print(f'  Q: {item["instruction"][:80]}{inp[:40]}')
    print(f'  A: {str(item["output"])[:80]}')


## 3. Verify the Bundled Source Tree

The image keeps the `MindSpeed-Core-MS`, `MindSpeed-LLM`, `MindSpeed`, `MSAdapter`, and `Megatron-LM` source trees in the same layout used by the official `MindSpeed-Core-MS/tests/scripts/set_path.sh`, so the notebook does not need to clone extra repositories.

The bundled source tree lives under `/opt/app-root/share/MindSpeed-Core-MS`, while `/opt/app-root/src` remains available for the workbench PVC, models, datasets, and training outputs.

This cell only performs directory checks and smoke tests for the official entry points. It does not modify the bundled source tree at notebook runtime.


In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)

repos = [
    ('MindSpeed-Core-MS', MINDSPEED_CORE_MS_PATH),
    ('MindSpeed-LLM', MINDSPEED_LLM_DIR),
    ('MindSpeed', MINDSPEED_DIR),
    ('MSAdapter', MSADAPTER_DIR),
    ('Megatron-LM', MEGATRON_DIR),
]

print('Bundled source tree:')
for name, repo_dir in repos:
    exists = repo_dir.exists()
    print(f'  [{name}] {repo_dir}: {"OK" if exists else "missing"}')
    assert exists, f'Bundled source tree missing: {repo_dir}'

script_checks = [
    ('Official weight conversion script', OFFICIAL_CONVERT_SCRIPT),
    ('Official MindSpore conversion entry', CONVERT_CKPT_ENTRY),
    ('Official instruction data script', OFFICIAL_PREPROCESS_SCRIPT),
    ('Data preprocessing entry', MINDSPEED_LLM_DIR / 'preprocess_data.py'),
    ('Official fine-tune script', OFFICIAL_TUNE_SCRIPT),
    ('Fine-tuning training entry', MINDSPEED_LLM_DIR / 'posttrain_gpt.py'),
]

print('
Official scripts and entry points:')
for name, script_path in script_checks:
    exists = script_path.exists()
    print(f'  [{name}] {script_path}: {"OK" if exists else "missing"}')
    assert exists, f'Required script missing: {script_path}'

repo_smoke_cmd = ' && '.join([
    f'cd {q(MINDSPEED_LLM_DIR)}',
    f'python {q(CONVERT_CKPT_ENTRY)} --load-model-type hf --help >/dev/null',
    'python ./preprocess_data.py --help >/dev/null',
    'python ./posttrain_gpt.py --help >/dev/null',
])
run_cmd(
    repo_smoke_cmd,
    cwd=MINDSPEED_LLM_DIR,
    step_name='repo-smoke-check',
    log_file=PRECHECK_LOG_DIR / 'repo_smoke_check.log',
)
print(f'Repository entry-point check log: {PRECHECK_LOG_DIR / "repo_smoke_check.log"}')
print('
Source tree checks passed!')


## 4. Convert HF Weights to MindSpeed/Mcore Format

Convert HuggingFace Qwen3-0.6B weights into the Mcore checkpoint format required by the MindSpore training flow. The first conversion usually takes a few minutes.

> If conversion hits device-side OOM, refer to the upstream quick start and run the conversion on the CPU side instead.


In [ ]:
MCORE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

conversion_marker = MCORE_WEIGHTS_DIR / 'latest_checkpointed_iteration.txt'
iter_dirs = sorted(MCORE_WEIGHTS_DIR.glob('iter_*'))
converted = conversion_marker.exists() and bool(iter_dirs)

if converted:
    print(f'Weights already exist, skipping conversion: {MCORE_WEIGHTS_DIR}')
    print(f'Latest checkpoint marker: {conversion_marker.read_text().strip()}')
else:
    convert_args = [
        '--use-mcore-models',
        '--model-type', 'GPT',
        '--load-model-type', 'hf',
        '--save-model-type', 'mg',
        '--target-tensor-parallel-size', str(TP),
        '--target-pipeline-parallel-size', str(PP),
        '--load-dir', str(HF_MODEL_DIR),
        '--save-dir', str(MCORE_WEIGHTS_DIR),
        '--tokenizer-model', str(HF_MODEL_DIR / 'tokenizer.json'),
        '--params-dtype', 'bf16',
        '--model-type-hf', 'qwen3',
        '--ai-framework', 'mindspore',
    ]
    if MODEL_SPEC:
        convert_args.extend(['--spec', *MODEL_SPEC.split()])

    convert_cmd = ' && '.join([
        f'cd {q(MINDSPEED_LLM_DIR)}',
        'export CUDA_DEVICE_MAX_CONNECTIONS=1',
        ' '.join(['python', q(CONVERT_CKPT_ENTRY), *[q(arg) for arg in convert_args]]),
    ])
    print('Converting weights through the official entry point...')
    run_cmd(
        convert_cmd,
        cwd=MINDSPEED_LLM_DIR,
        step_name='weight-convert',
        log_file=LOGS_DIR / 'convert_qwen3_0.6b_ms.log',
    )
    print('Weight conversion completed!')

print('
Converted checkpoint files:')
for p in sorted(MCORE_WEIGHTS_DIR.glob('*'))[:20]:
    print(f'  {p.name}')

assert conversion_marker.exists(), f'Checkpoint marker file not found: {conversion_marker}'
assert any(MCORE_WEIGHTS_DIR.glob('iter_*')), f'Converted iter_* checkpoint not found: {MCORE_WEIGHTS_DIR}'


## 5. Data Preprocessing

Convert Alpaca-formatted instruction data into the packed binary format required by the MindSpore Qwen3 fine-tuning flow.


In [ ]:
preprocess_cmd = ' && '.join([
    f'cd {q(MINDSPEED_LLM_DIR)}',
    f'mkdir -p {q(DATA_DIR)}',
    'python ./preprocess_data.py'
    f' --input {q(RAW_DATA_FILE)}'
    f' --tokenizer-name-or-path {q(HF_MODEL_DIR)}'
    f' --output-prefix {q(PROCESSED_DATA_PREFIX)}'
    f' --handler-name {HANDLER_NAME}'
    f' --tokenizer-type {TOKENIZER_TYPE}'
    ' --workers 4'
    ' --log-interval 1'
    f' --enable-thinking {ENABLE_THINKING}'
    f' --prompt-type {PROMPT_TYPE}',
])

print('Preprocessing data...')
run_cmd(
    preprocess_cmd,
    cwd=MINDSPEED_LLM_DIR,
    step_name='data-preprocess',
    log_file=LOGS_DIR / 'preprocess_qwen3_0.6b_ms.log',
)

expected_outputs = [
    DATA_DIR / 'alpaca_packed_attention_mask_document.bin',
    DATA_DIR / 'alpaca_packed_attention_mask_document.idx',
    DATA_DIR / 'alpaca_packed_input_ids_document.bin',
    DATA_DIR / 'alpaca_packed_input_ids_document.idx',
    DATA_DIR / 'alpaca_packed_labels_document.bin',
    DATA_DIR / 'alpaca_packed_labels_document.idx',
]

print('
Preprocessing output files:')
for f in expected_outputs:
    size_kb = f.stat().st_size / 1024 if f.exists() else 0
    print(f'  {f.name}: {"OK" if f.exists() else "missing"} ({size_kb:.1f} KB)')

missing = [str(f) for f in expected_outputs if not f.exists()]
assert not missing, f'Preprocessing output files were not generated: {missing}'
print('
Data preprocessing completed!')


## 6. Launch Fine-Tuning

Run full-parameter SFT for Qwen3-0.6B with the MindSpore backend. Training logs stream to the notebook.

> The current configuration `SEQ_LENGTH=2048, TRAIN_ITERS=100, MBS=2` targets 2x 910B 32G NPUs. For larger-scale training, increase `TRAIN_ITERS`, `SEQ_LENGTH`, and the dataset size.


In [ ]:
LOGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WORLD_SIZE = nproc * NNODES
DP = max(1, nproc // (TP * PP))
GBS = DP * MBS

train_log_file = LOGS_DIR / 'tune_qwen3_0.6b_ms.log'

env = ' && '.join([
    f'cd {q(MINDSPEED_LLM_DIR)}',
    'export CUDA_DEVICE_MAX_CONNECTIONS=1',
    'export PYTORCH_NPU_ALLOC_CONF=expandable_segments:True',
    f'mkdir -p {q(LOGS_DIR)}',
    f'mkdir -p {q(OUTPUT_DIR)}',
])

distributed = ' '.join([
    'msrun',
    f'--local_worker_num {nproc}',
    f'--worker_num {WORLD_SIZE}',
    f'--node_rank {NODE_RANK}',
    f'--master_addr {MASTER_ADDR}',
    f'--master_port {MASTER_PORT}',
    f'--log_dir={q(LOGS_DIR / "msrun")}',
    '--join=True',
    '--cluster_time_out=300',
])

model_args = ' '.join([
    '--use-mcore-models',
    f'--tensor-model-parallel-size {TP}',
    f'--pipeline-model-parallel-size {PP}',
    '--sequence-parallel',
    f'--spec {MODEL_SPEC}',
    '--kv-channels 128',
    '--qk-layernorm',
    '--use-flash-attn',
    '--num-layers 28',
    '--hidden-size 1024',
    '--use-rotary-position-embeddings',
    '--num-attention-heads 16',
    '--ffn-hidden-size 3072',
    '--max-position-embeddings 32768',
    f'--seq-length {SEQ_LENGTH}',
    '--make-vocab-size-divisible-by 1',
    '--padded-vocab-size 151936',
    '--rotary-base 1000000',
    '--disable-bias-linear',
    '--swiglu',
    '--tokenizer-type PretrainedFromHF',
    f'--tokenizer-name-or-path {q(HF_MODEL_DIR)}',
    '--normalization RMSNorm',
    '--position-embedding-type rope',
    '--norm-epsilon 1e-6',
    '--hidden-dropout 0',
    '--attention-dropout 0',
    '--no-gradient-accumulation-fusion',
    '--attention-softmax-in-fp32',
    '--exit-on-missing-checkpoint',
    '--no-masked-softmax-fusion',
    '--group-query-attention',
    '--num-query-groups 8',
    '--seed 42',
    '--bf16',
    '--transformer-impl local',
    '--ckpt-format msadapter',
])

train_args = ' '.join([
    f'--train-iters {TRAIN_ITERS}',
    f'--micro-batch-size {MBS}',
    f'--global-batch-size {GBS}',
    f'--lr {LR}',
    f'--min-lr {MIN_LR}',
    '--weight-decay 1e-1',
    '--lr-warmup-fraction 0.01',
    '--clip-grad 1.0',
    '--adam-beta1 0.9',
    '--adam-beta2 0.95',
    '--no-load-optim',
    '--no-load-rng',
])

data_args = ' '.join([
    f'--data-path {q(DATA_PATH)}',
    '--split 100,0,0',
])

output_args = ' '.join([
    '--log-interval 1',
    f'--save-interval {TRAIN_ITERS}',
    f'--eval-interval {TRAIN_ITERS}',
    '--eval-iters 0',
    '--log-throughput',
])

tune_args = ' '.join([
    '--finetune',
    '--stage sft',
    '--is-instruction-dataset',
    '--prompt-type qwen3',
    '--no-pad-to-seq-lengths',
])

cmd = (
    f'{env} && {distributed} posttrain_gpt.py '
    f'{model_args} {train_args} {data_args} {output_args} {tune_args} '
    f'--distributed-backend {DISTRIBUTED_BACKEND} '
    f'--load {q(MCORE_WEIGHTS_DIR)} '
    f'--save {q(OUTPUT_DIR)} '
    f'--ai-framework mindspore '
)

print(f'Training config: {nproc} NPU, TP={TP}, PP={PP}, DP={DP}, GBS={GBS}')
print(f'SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}, MASTER={MASTER_ADDR}:{MASTER_PORT}')
print(f'Log file: {train_log_file}')
print('
Starting fine-tuning...
')
run_cmd(
    cmd,
    cwd=MINDSPEED_LLM_DIR,
    step_name='train',
    log_file=train_log_file,
)
print(f'
Fine-tuning completed! Weights saved to: {OUTPUT_DIR}')


## 7. Validate Outputs


In [ ]:
log_file = LOGS_DIR / 'tune_qwen3_0.6b_ms.log'
latest_marker = OUTPUT_DIR / 'latest_checkpointed_iteration.txt'
iter_dirs = sorted(OUTPUT_DIR.glob('iter_*'))

print(f'Output directory: {OUTPUT_DIR}')
print(f'Log file: {log_file}')

assert OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()), f'No artifacts found in output directory: {OUTPUT_DIR}'

if latest_marker.exists():
    print(f'Latest checkpoint marker: {latest_marker.read_text().strip()}')
else:
    print('Latest checkpoint marker not found, listing all checkpoint artifacts instead.')

if iter_dirs:
    print('
Checkpoint iteration directories:')
    for d in iter_dirs:
        print(f'  {d.name}')

print('
Output artifacts (first 20 entries):')
for p in sorted(OUTPUT_DIR.rglob('*'))[:20]:
    if p.is_file():
        print(f'  {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'  {p.relative_to(OUTPUT_DIR)}/')

assert log_file.exists(), f'Training log was not generated: {log_file}'
print('
Validation passed!')


## Using a Real Dataset

Once validation passes, you can switch to a real dataset:

1. **Prepare the data**
   - Place a parquet dataset at `ALPACA_PARQUET`, or write Alpaca-formatted JSONL to `RAW_DATA_FILE`.
   - Each record should contain `instruction`, `input`, and `output` fields.

2. **Adjust the training scale**
   - Increase `SEQ_LENGTH` gradually based on available accelerator memory and target context length.
   - Increase `TRAIN_ITERS` based on dataset size.
   - Adjust `MBS` to match available memory.

3. **Re-convert weights when TP/PP changes**
   - The `MCORE_WEIGHTS_DIR` path includes `TP` and `PP` to prevent accidental reuse of stale converted weights.

4. **Adjust instruction preprocessing behavior**
   - The upstream example uses `ENABLE_THINKING=true` by default.
   - Change `ENABLE_THINKING` if your dataset or prompt style needs different behavior.

5. **Multi-node training**
   - Update `MASTER_ADDR`, `MASTER_PORT`, `NNODES`, and `NODE_RANK`, then rerun the training cell.

This notebook only validates checkpoint generation. Add inference steps later after the target image and upstream scripts provide a stable MindSpore inference path.
